# 💰 Medical Insurance Cost Predictor

Ye beginner-friendly **regression** based healthcare ML project hai jo predict karta hai ki kisi person ka **medical insurance charge (cost)** kitna hoga, uske personal aur lifestyle attributes (age, BMI, smoking habit, etc.) ke basis par.

**Steps in this notebook:**
1. Dataset generate/load karna
2. Exploratory Data Analysis (EDA)
3. Data preprocessing (encoding categorical features)
4. Model training (Linear Regression + Random Forest Regressor)
5. Model evaluation (R², MAE, RMSE)
6. Prediction on a new person's data

> Note: Yahan realistic **synthetic dataset** generate kiya gaya hai (famous "Medical Cost Personal Dataset" jaisa structure — age, sex, bmi, children, smoker, region, charges), taaki notebook fully offline chal jaye. Agar aapke paas Kaggle ka real `insurance.csv` hai, to Section 2 mein bas usse load kar sakte ho — baaki code same rahega.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
np.random.seed(42)
print("Libraries loaded successfully ✅")


## 2. Dataset Generate Karna

Neeche 1000 logon ka synthetic insurance data generate ho raha hai jisme columns hain:

| Column | Meaning |
|---|---|
| age | Person's age |
| sex | male / female |
| bmi | Body Mass Index |
| children | Number of dependents |
| smoker | yes / no |
| region | northeast / northwest / southeast / southwest |
| charges | Insurance cost (target variable) |

Agar aapke paas real dataset (`insurance.csv`) hai, to yahan `pd.read_csv("insurance.csv")` use kar sakte ho — baaki notebook bina change ke chalega.

In [ ]:
n = 1000

age = np.random.randint(18, 65, n)
sex = np.random.choice(['male', 'female'], n)
bmi = np.random.normal(30, 6, n).clip(15, 55)
children = np.random.choice([0, 1, 2, 3, 4, 5], n, p=[0.35, 0.25, 0.2, 0.12, 0.05, 0.03])
smoker = np.random.choice(['yes', 'no'], n, p=[0.2, 0.8])
region = np.random.choice(['northeast', 'northwest', 'southeast', 'southwest'], n)

# Charges ek realistic formula + noise se banaya gaya hai
base_cost = (
    250 * age +
    320 * bmi +
    400 * children +
    (smoker == 'yes') * 22000 +
    np.random.normal(0, 2000, n)
)
charges = np.clip(base_cost, 1200, None).round(2)

df = pd.DataFrame({
    "age": age,
    "sex": sex,
    "bmi": bmi.round(1),
    "children": children,
    "smoker": smoker,
    "region": region,
    "charges": charges
})

df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Shape of dataset:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
df.describe()


In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df['charges'], bins=30, kde=True, color='teal')
plt.title('Distribution of Insurance Charges')
plt.xlabel('Charges')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='smoker', y='charges', data=df, palette='Set2', hue='smoker', legend=False)
plt.title('Charges: Smoker vs Non-Smoker')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x='age', y='charges', hue='smoker', data=df, palette='Set1')
plt.title('Age vs Charges (colored by smoking status)')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df, palette='Set1')
plt.title('BMI vs Charges (colored by smoking status)')
plt.show()


## 4. Data Preprocessing

In [ ]:
df_encoded = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)
df_encoded.head()


In [ ]:
X = df_encoded.drop('charges', axis=1)
y = df_encoded['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 5. Model Training

In [ ]:
# --- Linear Regression ---
lin_model = LinearRegression()
lin_model.fit(X_train_scaled, y_train)
lin_preds = lin_model.predict(X_test_scaled)

# --- Random Forest Regressor ---
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("Models trained successfully ✅")


## 6. Model Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- {name} ---")
    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R2   : {r2:.3f}\n")

evaluate("Linear Regression", y_test, lin_preds)
evaluate("Random Forest Regressor", y_test, rf_preds)


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, rf_preds, alpha=0.5, color='darkorange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Charges')
plt.ylabel('Predicted Charges')
plt.title('Random Forest: Actual vs Predicted Charges')
plt.show()


In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importance.values, y=importance.index, hue=importance.index, palette='viridis', legend=False)
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance Score")
plt.show()


## 7. Predict on New Person's Data

Neeche ek naye person ka data daal ke dekhte hain model kitna insurance cost predict karta hai.

In [ ]:
new_person = pd.DataFrame([{
    "age": 35,
    "bmi": 28.5,
    "children": 2,
    "sex_male": 1,
    "smoker_yes": 1,
    "region_northwest": 0,
    "region_southeast": 1,
    "region_southwest": 0
}])

# Training columns ke order se match karna zaroori hai
new_person = new_person.reindex(columns=X.columns, fill_value=0)

predicted_cost = rf_model.predict(new_person)[0]
print(f"Predicted Insurance Cost: ${predicted_cost:,.2f}")


## 8. Conclusion & Next Steps

- **Smoking status** aur **age**/**BMI** insurance cost pe sabse zyada impact daalte hain.
- Random Forest generally Linear Regression se better fit karta hai kyunki data mein non-linear interactions hain (jaise smoking + BMI ka combined effect).

**Next steps for improvement:**
- Real dataset use karo (Kaggle: "Medical Cost Personal Datasets")
- Hyperparameter tuning (GridSearchCV)
- Try Gradient Boosting / XGBoost Regressor
- Cross-validation add karo
- Streamlit se ek simple "Insurance Cost Calculator" web app bana do

Ye project bhi GitHub par upload karke portfolio mein add kar sakte ho! 🚀